### imports and pandas settings

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

In [2]:
pd.set_option("display.max_rows", None)

### set the path to data file

In [3]:
load_dotenv()

CF_OUTPUTS = Path(os.getenv("CF_OUTPUTS"))
print(CF_OUTPUTS)
print(CF_OUTPUTS.is_dir())

/home/dyretna/Dokument/Code/GitHub/nightingale_projects/cf_bench/cf_outputs
True


In [4]:
batch_data = CF_OUTPUTS / "predictors_vs_threshold" / "SMOTE" / "base_predictors" / "XGBoost_thres0.9_2026-05-11" / "annotated.csv"

### load data 
convert to string to fillna with whitespace
we drop:  
- target_risk (half of "risk_before") , and 
- "hltprhc"

we rename "original" in df_id to "xin" (HL standards)

In [5]:
# Load CSV
batch_df = pd.read_csv(batch_data)

In [6]:
# Feature columns
feature_cols = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "bmi", "dosprt"]

# dtypes
int_columns = ["etfruit", "eatveg", "cgtsmok", "alcfreq", "slprl", "paccnois", "dosprt", "hltprhc", "Nchanged"]
float_columns = ["bmi", "gower_distance", "risk_before", "predicted_risk_after", "cf_gen_time_sec"]


# Convert and fill NaN with empty strings
batch_df[int_columns] = batch_df[int_columns].astype("Int64").astype("string").fillna("")
batch_df[float_columns] = batch_df[float_columns].round(4).astype("string").fillna("")
batch_df["valid"] = batch_df["valid"].astype("string").fillna("")

In [7]:
batch_df = batch_df.drop(columns=["hltprhc", "target_risk"])
batch_df["cf_id"] = batch_df["cf_id"].replace({"original": "xin"})

### All CFs

In [8]:
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.1,,,,,
1,0,cf_1,1,,,,,,18.9388,,,0.1268,2,False,0.0274,0.0324
2,0,cf_2,3,,,,,,18.7707,,,0.0497,2,False,0.0274,0.2389
3,0,cf_3,,,,,,,15.6339,,,0.125,1,True,0.0274,0.0124
4,0,cf_4,,,6,,,,17.5641,,,0.178,2,False,0.0274,0.0284
5,0,cf_5,,,,,,,17.2158,5,,0.1553,2,True,0.0274,0.0021
6,0,cf_6,,,,6,,,17.1306,,,0.1525,2,False,0.0274,0.1427
7,0,cf_7,3,,,,,,18.9745,7,,0.1671,3,False,0.0274,0.2063
8,0,cf_8,,,,7,,,18.9745,7,,0.2504,3,False,0.0274,0.0698
9,0,cf_9,,,5,,,,18.9619,2,,0.12,3,False,0.0274,0.0244


### Filtering Valid CFs

In [9]:
from cf_bench.utils import filter_valid_cfs

batch_df = filter_valid_cfs(batch_df)
batch_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.1,,,,,
9,0,cf_3,,,,,,,15.6339,,,0.125,1,True,0.0274,0.0124
10,0,cf_5,,,,,,,17.2158,5,,0.1553,2,True,0.0274,0.0021
1,1,xin,3,4,1,2,3,0,22.3757,0,0.99,,,,,
11,1,cf_1,,,,,1,,22.3757,1,,0.1609,3,True,0.6924,0.2375
12,1,cf_2,2,,,,1,,22.3757,,,0.2546,3,True,0.6924,0.1514
13,1,cf_3,,3,,,1,,22.3757,4,,0.3796,4,True,0.6924,0.182
2,2,xin,5,3,1,1,4,0,22.694,7,1.2,,,,,
14,2,cf_4,1,,,4,1,,22.68,,,0.3255,4,True,0.0454,0.0127
3,3,xin,3,3,6,6,2,0,24.3418,1,1.05,,,,,


### select one CF

In [10]:
from cf_bench.utils import select_one_cf_per_query

# Select one CF per observation (prefers valid CFs without violations)
single_cf_df = select_one_cf_per_query(batch_df, prefer_no_violations=True)
single_cf_df

,query_index,cf_id,etfruit,eatveg,cgtsmok,alcfreq,slprl,paccnois,bmi,dosprt,cf_gen_time_sec,gower_distance,Nchanged,valid,risk_before,predicted_risk_after
0,0,xin,4,3,3,4,2,0,18.9866,0,1.1,,,,,
9,0,cf_5,,,,,,,17.2158,5,,0.1553,2,True,0.0274,0.0021
1,1,xin,3,4,1,2,3,0,22.3757,0,0.99,,,,,
10,1,cf_1,,,,,1,,22.3757,1,,0.1609,3,True,0.6924,0.2375
2,2,xin,5,3,1,1,4,0,22.694,7,1.2,,,,,
11,2,cf_4,1,,,4,1,,22.68,,,0.3255,4,True,0.0454,0.0127
3,3,xin,3,3,6,6,2,0,24.3418,1,1.05,,,,,
12,3,cf_1,,,,,,,23.4429,,,0.125,1,False,0.4508,0.2489
4,4,xin,5,4,2,7,1,0,21.2585,3,1.18,,,,,
13,4,cf_1,,,,,,,21.2585,,,0.1118,1,False,0.0149,0.0149


In [11]:
from cf_bench.dice_adapters import DiceRecommender

recommender = DiceRecommender()

# Format a single query
for idx in range(0, 10):
    print(recommender.format_query(batch_df, query_index=idx))


=== Query index '0' ===
Task / Target: hltprhc
Query instance index: 0

Original instance:
  etfruit: 4
  eatveg: 3
  cgtsmok: 3
  alcfreq: 4
  slprl: 2
  paccnois: 0
  bmi: 18.9866
  dosprt: 0


=== Counterfactuals ===

--- cf_3 ---
Predicted risk: 0.0124
Valid: True
Changes:
  bmi: 18.9866 → 15.6339

--- cf_5 ---
Predicted risk: 0.0021
Valid: True
Changes:
  bmi: 18.9866 → 17.2158
  dosprt: 0 → 5


=== Query index '1' ===
Task / Target: hltprhc
Query instance index: 1

Original instance:
  etfruit: 3
  eatveg: 4
  cgtsmok: 1
  alcfreq: 2
  slprl: 3
  paccnois: 0
  bmi: 22.3757
  dosprt: 0


=== Counterfactuals ===

--- cf_1 ---
Predicted risk: 0.2375
Valid: True
Changes:
  slprl: 3 → 1
  dosprt: 0 → 1

--- cf_2 ---
Predicted risk: 0.1514
Valid: True
Changes:
  etfruit: 3 → 2
  slprl: 3 → 1

--- cf_3 ---
Predicted risk: 0.182
Valid: True
Changes:
  eatveg: 4 → 3
  slprl: 3 → 1
  dosprt: 0 → 4


=== Query index '2' ===
Task / Target: hltprhc
Query instance index: 2

Original instance: